# Преобразование XLSB в CSV

Поместите этот блокнот рядом с папкой `download_reestr` и запустите ячейки по порядку. Результаты сохраняются в `download_reestr_csv_misha` с сохранением вложенных папок.

Преобразуется первый лист каждого файла. Разделитель — запятая, кодировка — UTF-8 с BOM. Исходные файлы сохраняются. Повторный запуск перезаписывает созданные CSV.


In [ ]:
%pip install pandas pyxlsb


In [ ]:
from pathlib import Path
import pandas as pd

source_dir = Path("download_reestr")
target_dir = Path("download_reestr_csv_misha")

if not source_dir.is_dir():
    raise FileNotFoundError(f"Не найдена папка: {source_dir.resolve()}")

files = sorted(
    path for path in source_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() == ".xlsb"
    and not path.name.startswith("~$")
)

print(f"Найдено файлов: {len(files)}")
converted = 0
errors = []

for number, source_file in enumerate(files, start=1):
    relative_path = source_file.relative_to(source_dir)
    target_file = (target_dir / relative_path).with_suffix(".csv")
    target_file.parent.mkdir(parents=True, exist_ok=True)

    try:
        df = pd.read_excel(source_file, engine="pyxlsb", sheet_name=0)
        df.to_csv(target_file, sep=",", index=False, encoding="utf-8-sig")
        converted += 1
        print(f"[{number}/{len(files)}] Готово: {relative_path}")
    except Exception as error:
        errors.append((str(relative_path), str(error)))
        print(f"[{number}/{len(files)}] Ошибка: {relative_path} — {error}")

print(f"\nПреобразовано: {converted} из {len(files)}")
print(f"Ошибок: {len(errors)}")
print(f"Результат: {target_dir.resolve()}")
